# SpaceX Falcon 9 — Web Scraping Wikipedia
Parse the *List of Falcon 9 and Falcon Heavy launches* HTML tables with BeautifulSoup.

In [ ]:
!pip install beautifulsoup4 -q
import sys, requests, re, unicodedata
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]
def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out
def landing_status(table_cells):
    return [i for i in table_cells.strings][0]
def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg"); new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass
def extract_column_from_header(row):
    if row.br: row.br.extract()
    if row.a: row.a.extract()
    if row.sup: row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not (colunm_name.strip().isdigit()):
        return colunm_name.strip()

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title)

In [ ]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

In [ ]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']
for k in ['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome',
          'Version Booster', 'Booster landing', 'Date', 'Time']:
    launch_dict[k] = []

In [ ]:
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip(); flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])
            bv = booster_version(row[1])
            if not bv: bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(row[2].a.string)
            launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].text.strip())
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].text.strip())
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].text.strip())
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))
print(extracted_row)

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(df.shape)
df.head()

**Result:** 121 launch records scraped from the Wikipedia tables.

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)